In [ ]:
# Copyright (c) 2025 Microsoft Corporation.

import sys

sys.path.insert(1, "../../../")

In [ ]:
%load_ext dotenv
%dotenv

In [ ]:
import logging
import os

import pandas as pd
import tiktoken
from pydantic import SecretStr

from benchmark_qed.autod.data_processor.embedding import TextEmbedder
from benchmark_qed.autod.io.text_unit import load_text_units
from benchmark_qed.autoq.io.question import (
    load_questions,
    save_questions,
)
from benchmark_qed.config.llm_config import LLMConfig, LLMProvider
from benchmark_qed.llm.factory import ModelFactory

logging.basicConfig(level=logging.INFO)

if logging.getLogger("httpx") is not None:
    logging.getLogger("httpx").setLevel(logging.ERROR)

# Experiment 11f: AutoQ Data-Global Question Generation

This notebook demonstrates **data-global question generation** using AutoQ on the AP News dataset. Data-global questions target general themes, patterns, and concerns that span the entire corpus, synthesizing insights from multiple local questions.

Data-global questions are generated in two stages:
1. **Data-Local Question Generation**: Generate candidate local questions grounded in specific text passages. These local questions capture detailed, factual information about individual events or entities.
2. **Data-Global Question Generation**: Aggregate local questions by abstract category (e.g., themes, concerns) and generate higher-level global questions that synthesize across multiple local questions.

The output global questions can be used as evaluation queries for RAG systems that require broad, thematic reasoning over a corpus (e.g., graph-based RAG methods).

This experiment builds on the data-local pipeline demonstrated in Experiment 11b, adding the global synthesis step.

## Configs

In [ ]:
# DATA CONFIGS
INPUT_DATA_PATH = "../../datasets/AP_news/raw_data"
OUTPUT_DATA_PATH = "../../output/AP_news/processed_data"
OUTPUT_QUESTIONS_PATH = "../../output/AP_news/questions"
TEXT_COLUMN = "body_nitf"
METADATA_COLUMNS = ["headline", "firstcreated"]
FILE_ENCODING = "utf-8-sig"

# tokenizer used for chunking documents into text units
ENCODING_MODEL = "o200k_base"
CHUNK_SIZE = 600
CHUNK_OVERLAP = 100

# DATA SAMPLING CONFIGS
# These configs control the breadth and depth of the selected data sample.
# Adjust these parameters based on your data size and the number of questions to be generated.
# The final sample size will be NUM_CLUSTERS * NUM_SAMPLES_PER_CLUSTER
NUM_CLUSTERS = 20
NUM_SAMPLES_PER_CLUSTER = 10
RANDOM_SEED = 42

# QUESTION GENERATION CONFIGS
NUM_QUESTIONS = 10  # Number of final questions to select
OVERSAMPLE_FACTOR = 2.0  # Factor by which to overgenerate candidate questions

In [ ]:
# MODEL CONFIGS
API_KEY = SecretStr(os.getenv("OPENAI_API_KEY", ""))
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-4.1"
LLM_PARAMS = {
    "temperature": 0.0,
    "seed": 42,
}  # adjust this based on your model. For example, some reasoning models do not support temperature settings
CONCURRENT_REQUESTS = (
    8  # Control for request concurrency. Adjust this based on your model capacity.
)

# ASSERTION GENERATION CONFIGS
MAX_ASSERTIONS = 20  # Maximum number of assertions per question (set to 0 to disable)
BATCH_SIZE = 100  # Batch size for processing claims in global assertion generation
MAX_DATA_TOKENS = (
    32000  # Maximum input data tokens for the reduce step in global assertions
)
ENABLE_VALIDATION = True  # Set to True to validate assertions against sources
MIN_VALIDATION_SCORE = 3  # Minimum score (1-5) for grounding, relevance, verifiability

# Parallelism for assertion generation (adjust based on your model rate limits)
CONCURRENT_LOCAL_QUESTIONS = 8  # Questions to process in parallel for local assertions
CONCURRENT_GLOBAL_QUESTIONS = 2  # Questions to process in parallel for global assertions (lower due to internal parallelism, set to 1 for sequential)

text_embedder = TextEmbedder(
    ModelFactory.create_embedding_model(
        LLMConfig(
            model=EMBEDDING_MODEL,
            api_key=API_KEY,
            llm_provider=LLMProvider.OpenAIEmbedding,
        )
    )
)
llm = ModelFactory.create_chat_model(
    model_config=LLMConfig(
        model=LLM_MODEL,
        api_key=API_KEY,
        llm_provider=LLMProvider.OpenAIChat,
        call_args=LLM_PARAMS,
    )
)
token_encoder = tiktoken.get_encoding(ENCODING_MODEL)

## Step 1: Data Sampling

Load documents from the input folder, chunk them into text units, embed all text units, and sample a clustered subset. These clustered sample texts ground the local question generation process.

If you have previously run this sampling step, you can skip it and load the saved sample from disk in the next section.

In [ ]:
from benchmark_qed.autod.sampler.sample_gen import acreate_clustered_sample

clustered_sample = await acreate_clustered_sample(
    input_path=INPUT_DATA_PATH,
    output_path=OUTPUT_DATA_PATH,
    text_embedder=text_embedder,
    num_clusters=NUM_CLUSTERS,
    num_samples_per_cluster=NUM_SAMPLES_PER_CLUSTER,
    input_type="json",
    text_tag=TEXT_COLUMN,
    metadata_tags=METADATA_COLUMNS,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    file_encoding=FILE_ENCODING,
    token_encoding=ENCODING_MODEL,
    random_seed=RANDOM_SEED,
)
print(
    f"Sampled {len(clustered_sample.sample_texts)} samples from {len(clustered_sample.text_units)} text units in {len(clustered_sample.documents)} documents."
)

## Step 2: Data-Local Question Generation

Generate candidate local questions grounded in specific text passages from the corpus. The local questions capture detailed, fact-grounded information about individual events or entities. They also annotate each question with **abstract categories** (themes) that will be used to group questions for global synthesis.

The local question generation process:
1. Groups sample texts by cluster
2. Generates candidate questions for each cluster using a two-step (extract + expand) chain
3. Ranks and selects the best `NUM_QUESTIONS` local questions based on similarity and coverage metrics
4. Optionally generates assertions for the selected local questions

All **candidate** local questions (not just the selected ones) are saved to disk so that the global question generation step can draw from the full pool of local questions.

In [ ]:
from benchmark_qed.autoq.config import (
    AssertionConfig,
    GlobalAssertionConfig,
    LocalAssertionConfig,
)
from benchmark_qed.autoq.question_gen.data_questions.local_question_gen import (
    DataLocalQuestionGen,
)

# Load clustered text sample from disk (result from Step 1).
# You can also use clustered_sample.sample_texts directly if Step 1 was just run.
sample_texts_df = pd.read_parquet(f"{OUTPUT_DATA_PATH}/sample_texts.parquet")
sample_texts = load_text_units(df=sample_texts_df)

# Configure assertion generation
# Set max_assertions to an integer to limit assertions, 0 to disable, None for unlimited
local_assertion_config = LocalAssertionConfig(
    max_assertions=MAX_ASSERTIONS,
    enable_validation=ENABLE_VALIDATION,
    min_validation_score=MIN_VALIDATION_SCORE,
    concurrent_llm_calls=CONCURRENT_REQUESTS,
    max_concurrent_questions=CONCURRENT_LOCAL_QUESTIONS,
)
global_assertion_config = GlobalAssertionConfig(
    max_assertions=MAX_ASSERTIONS,
    enable_validation=ENABLE_VALIDATION,
    min_validation_score=MIN_VALIDATION_SCORE,
    batch_size=BATCH_SIZE,
    max_data_tokens=MAX_DATA_TOKENS,
    concurrent_llm_calls=CONCURRENT_REQUESTS,
    max_concurrent_questions=CONCURRENT_GLOBAL_QUESTIONS,
)
assertion_config = AssertionConfig(
    local=local_assertion_config,
    **{
        "global": global_assertion_config
    },  # Use dict unpacking since "global" is a keyword
)

data_local_generator = DataLocalQuestionGen(
    llm=llm,
    llm_params=LLM_PARAMS,
    text_embedder=text_embedder,
    text_units=sample_texts,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    random_seed=RANDOM_SEED,
    assertion_config=assertion_config,
)

data_local_question_results = await data_local_generator.agenerate(
    num_questions=NUM_QUESTIONS,
    oversample_factor=OVERSAMPLE_FACTOR,
)

print(
    f"Generated {len(data_local_question_results.candidate_questions)} candidate local questions."
)
print(
    f"Selected {len(data_local_question_results.selected_questions)} final local questions."
)

# Save both candidate questions and the final selected questions.
# NOTE: We save all candidate questions (not just selected) so the global step
# can draw from the full pool of annotated local questions.
save_questions(
    data_local_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions",
)
save_questions(
    data_local_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "selected_questions_text",
    question_text_only=True,
)
save_questions(
    data_local_question_results.candidate_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/",
    "candidate_questions",
)

print("\nSelected local questions:")
for i, q in enumerate(data_local_question_results.selected_questions):
    print(f"{i + 1}. {q.text}")

## Step 3: Data-Global Question Generation

Generate global questions that synthesize across multiple local questions. The global question generation process:
1. Groups candidate local questions by their **abstract categories** (themes annotated during local question generation)
2. For each category with enough local questions, generates candidate global questions that reflect broader themes
3. Extracts relevant claims from the local questions to support each global question
4. Ranks and selects the best `NUM_QUESTIONS` global questions based on claim counts and reference coverage
5. Optionally generates assertions for the selected global questions

**Important**: The global generator uses all **candidate** local questions (not just the selected ones) to ensure a rich pool for category grouping and synthesis. Loading from disk allows you to reuse previously generated local questions without re-running Step 2.

In [ ]:
from benchmark_qed.autoq.question_gen.data_questions.global_question_gen import (
    DataGlobalQuestionGen,
)

# Load all candidate local questions from disk (result from Step 2).
# Using candidate questions (not just selected) gives the global generator a
# larger pool of theme-annotated questions to group and synthesize from.
# You can also use data_local_question_results.candidate_questions directly
# if Step 2 was just run.
local_questions = load_questions(
    f"{OUTPUT_QUESTIONS_PATH}/data_local_questions/candidate_questions.json"
)
print(f"Loaded {len(local_questions)} candidate local questions for global synthesis.")

# Reuse the same assertion config from the local question generation step
data_global_generator = DataGlobalQuestionGen(
    llm=llm,
    llm_params=LLM_PARAMS,
    text_embedder=text_embedder,
    token_encoder=token_encoder,
    local_questions=local_questions,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    random_seed=RANDOM_SEED,
    assertion_config=assertion_config,
)

data_global_question_results = await data_global_generator.agenerate(
    num_questions=NUM_QUESTIONS,
    oversample_factor=OVERSAMPLE_FACTOR,
)

print(
    f"Generated {len(data_global_question_results.candidate_questions)} candidate global questions."
)
print(
    f"Selected {len(data_global_question_results.selected_questions)} final global questions."
)

In [ ]:
# Save both candidate questions and the final selected questions
save_questions(
    data_global_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "selected_questions",
)
save_questions(
    data_global_question_results.selected_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "selected_questions_text",
    question_text_only=True,
)
save_questions(
    data_global_question_results.candidate_questions,
    f"{OUTPUT_QUESTIONS_PATH}/data_global_questions/",
    "candidate_questions",
)

print("Saved selected and candidate global questions to disk.")
print("\nSelected global questions:")
for i, q in enumerate(data_global_question_results.selected_questions):
    print(f"{i + 1}. {q.text}")